# Maritime AIS Data Analysis (01/08/2025)


## Overview
This notebook loads AIS ship tracking data from NOAA, cleans it, and performs two analyses:
1. When boats are out (hourly unique vessel counts)
2. Engine activity for vessel Henry Hudson (MMSI: 366651000)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Load dataset (compressed CSV)
url = "https://noaaocm.blob.core.windows.net/ais/csv2/csv2025/ais-2025-01-08.csv.zst"
df = pd.read_csv(url)

df.head()


In [ ]:

# Convert base_date_time to datetime and set as index
df['base_date_time'] = pd.to_datetime(df['base_date_time'])
df = df.set_index('base_date_time')

# Sort index
df = df.sort_index()

df.info()


In [ ]:

# Clean data: remove rows where sog is null
df = df[df['sog'].notna()]

df.shape


## First Analysis: When are boats out?

In [ ]:

# Extract hour
df['hour'] = df.index.hour

# Count unique vessels per hour
hourly_counts = df.groupby('hour')['mmsi'].nunique()

# Plot
plt.figure()
hourly_counts.plot()
plt.xlabel("Hour of Day (UTC)")
plt.ylabel("Number of Unique Vessels")
plt.title("Unique Vessels by Hour")
plt.show()

# Identify max and min
max_hour = hourly_counts.idxmax()
min_hour = hourly_counts.idxmin()

max_hour, min_hour



### Interpretation
- The busiest hour (UTC) is shown above.
- The least busy hour (UTC) is shown above.
- Typically, vessel activity increases during daylight hours and decreases overnight.


## Second Analysis: Henry Hudson Engine Activity

In [ ]:

# Filter for Henry Hudson
henry = df[df['mmsi'] == 366651000].copy()

# Ensure sorted
henry = henry.sort_index()

# Acceleration (difference in sog)
henry['acceleration'] = henry['sog'].diff()

# Duration between timestamps
henry['duration'] = henry.index.to_series().diff()

henry.head()


In [ ]:

# Filter where acceleration > 0
active = henry[henry['acceleration'] > 0]

# Total engine active time
total_active_time = active['duration'].sum()

total_active_time



### Result
The total engine active time for the Henry Hudson is shown above.
